In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

#Work out the project folder depending on where the notebook is being run from
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "data" / "raw").exists():
    PROJECT_DIR = CURRENT_DIR
else:
    PROJECT_DIR = CURRENT_DIR.parent

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"

print("Current folder:", CURRENT_DIR)
print("Project folder:", PROJECT_DIR)
print("Raw data folder:", RAW_DATA_DIR)

#Loading all CIC-IDS2017 CSV files from the raw data folder
csv_files = sorted(RAW_DATA_DIR.rglob("*.csv"))

print("CSV files found:", len(csv_files))
for file in csv_files:
    print(file)

if len(csv_files) == 0:
    raise FileNotFoundError(
        f"No CSV files found in {RAW_DATA_DIR}. Check that the CIC-IDS2017 CSV files are inside data/raw."
    )

#Reading each file and keeping the source file name for later day/file analysis
dataframes = []

for file in csv_files:
    print("Loading:", file.name)
    temp_df = pd.read_csv(file, low_memory=False)
    temp_df.columns = temp_df.columns.str.strip()
    temp_df["source_file"] = file.name
    dataframes.append(temp_df)

df = pd.concat(dataframes, ignore_index=True)

print("Combined shape:", df.shape)

#Converting the original attack labels into a binary BENIGN/ATTACK target
df["binary_label"] = np.where(df["Label"] == "BENIGN", "BENIGN", "ATTACK")

#Replacing infinite values before removing rows with missing values
df = df.replace([np.inf, -np.inf], np.nan)

missing_before = df.isna().sum()
missing_before = missing_before[missing_before > 0]

print("\nMissing values before cleaning:")
print(missing_before)

df_clean = df.dropna().copy()

print("\nOriginal shape:", df.shape)
print("Cleaned shape:", df_clean.shape)
print("Rows removed:", df.shape[0] - df_clean.shape[0])

print("\nFinal label counts:")
print(df_clean["Label"].value_counts())

print("\nFinal binary counts:")
print(df_clean["binary_label"].value_counts())

Current folder: /Users/tvishaprasad/Documents/AICS_CICIDS2017_Project/notebooks
Project folder: /Users/tvishaprasad/Documents/AICS_CICIDS2017_Project
Raw data folder: /Users/tvishaprasad/Documents/AICS_CICIDS2017_Project/data/raw
CSV files found: 8
/Users/tvishaprasad/Documents/AICS_CICIDS2017_Project/data/raw/MachineLearningCVE/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
/Users/tvishaprasad/Documents/AICS_CICIDS2017_Project/data/raw/MachineLearningCVE/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
/Users/tvishaprasad/Documents/AICS_CICIDS2017_Project/data/raw/MachineLearningCVE/Friday-WorkingHours-Morning.pcap_ISCX.csv
/Users/tvishaprasad/Documents/AICS_CICIDS2017_Project/data/raw/MachineLearningCVE/Monday-WorkingHours.pcap_ISCX.csv
/Users/tvishaprasad/Documents/AICS_CICIDS2017_Project/data/raw/MachineLearningCVE/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
/Users/tvishaprasad/Documents/AICS_CICIDS2017_Project/data/raw/MachineLearningCVE/Thursday-WorkingHours

In [ ]:
#Removing exact duplicate records before splitting to reduce memorisation risk

feature_columns = [
    col for col in df_clean.columns
    if col not in ["Label", "binary_label", "source_file"]
]

dedup_subset = feature_columns + ["binary_label"]

original_rows = df_clean.shape[0]
duplicate_rows = df_clean.duplicated(subset=dedup_subset).sum()

df_model = df_clean.drop_duplicates(subset=dedup_subset).copy()

dedup_rows = df_model.shape[0]
removed_rows = original_rows - dedup_rows

print("Original cleaned rows:", original_rows)
print("Exact duplicate rows found:", duplicate_rows)
print("Rows after deduplication:", dedup_rows)
print("Duplicate rows removed:", removed_rows)
print("Percentage removed:", round((removed_rows / original_rows) * 100, 3))

print("\nBinary distribution after deduplication:")
print(df_model["binary_label"].value_counts())
print(df_model["binary_label"].value_counts(normalize=True).mul(100).round(2))

In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

X = df_model[feature_columns]
y = df_model["binary_label"]

print("Feature matrix:", X.shape)
print("Target:", y.shape)

print("\nClass distribution:")
print(y.value_counts())
print(y.value_counts(normalize=True).mul(100).round(2))

#First split: keep 20% aside as the final untouched test set
X_train_val, X_test_final, y_train_val, y_test_final = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

#Second split: split the remaining 80% into training and validation sets
X_train_final, X_val_final, y_train_final, y_val_final = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_train_val
)

print("\nTraining set:", X_train_final.shape)
print("Validation set:", X_val_final.shape)
print("Final test set:", X_test_final.shape)

print("\nTraining distribution:")
print(y_train_final.value_counts(normalize=True).mul(100).round(2))

print("\nValidation distribution:")
print(y_val_final.value_counts(normalize=True).mul(100).round(2))

print("\nFinal test distribution:")
print(y_test_final.value_counts(normalize=True).mul(100).round(2))

In [ ]:
import matplotlib.pyplot as plt
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    average_precision_score,
    roc_auc_score
)

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier

#Creating folders for saved results, figures and trained models
RESULTS_DIR = PROJECT_DIR / "outputs" / "results"
FIGURE_DIR = PROJECT_DIR / "outputs" / "figures"
MODEL_DIR = PROJECT_DIR / "models"

for folder in [RESULTS_DIR, FIGURE_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

#Scaling is placed inside the pipeline so it is fitted only on training data
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), feature_columns)
    ]
)

def get_attack_scores(model, X_data):
    attack_index = list(model.classes_).index("ATTACK")
    return model.predict_proba(X_data)[:, attack_index]

def evaluate_predictions(y_true, y_pred, y_score=None, dataset_name="Evaluation"):
    results = {
        "dataset": dataset_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "attack_precision": precision_score(y_true, y_pred, pos_label="ATTACK", zero_division=0),
        "attack_recall": recall_score(y_true, y_pred, pos_label="ATTACK", zero_division=0),
        "attack_f1": f1_score(y_true, y_pred, pos_label="ATTACK", zero_division=0)
    }

    if y_score is not None:
        y_binary = (y_true == "ATTACK").astype(int)
        results["pr_auc"] = average_precision_score(y_binary, y_score)
        results["roc_auc"] = roc_auc_score(y_binary, y_score)

    print(dataset_name)
    print("-" * 60)
    for key, value in results.items():
        if key != "dataset":
            print(f"{key}: {value:.4f}")

    print("\nClassification report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    return results

print("Setup complete.")
print("Results folder:", RESULTS_DIR.resolve())
print("Figures folder:", FIGURE_DIR.resolve())
print("Models folder:", MODEL_DIR.resolve())

In [ ]:
#Comparing several baseline and tree-based models on the validation set
models = {
    "Dummy Baseline": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", DummyClassifier(strategy="most_frequent"))
    ]),

    "Logistic Regression": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=500,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),

    "Decision Tree": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),

    "Random Forest": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),

    "Extra Trees": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", ExtraTreesClassifier(
            n_estimators=100,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),

    "HistGradientBoosting": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", HistGradientBoostingClassifier(
            random_state=RANDOM_STATE
        ))
    ])
}

validation_results = []

for model_name, model in models.items():
    print("\n" + "=" * 80)
    print(f"Training {model_name}")
    print("=" * 80)

    model.fit(X_train_final, y_train_final)

    val_pred = model.predict(X_val_final)

    if hasattr(model.named_steps["classifier"], "predict_proba"):
        val_scores = get_attack_scores(model, X_val_final)
    else:
        val_scores = None

    result = evaluate_predictions(
        y_val_final,
        val_pred,
        y_score=val_scores,
        dataset_name=f"Validation - {model_name}"
    )

    result["model"] = model_name
    validation_results.append(result)

validation_results_df = pd.DataFrame(validation_results).sort_values(
    by="attack_f1",
    ascending=False
)

validation_results_df.to_csv(
    RESULTS_DIR / "validation_model_comparison_clean.csv",
    index=False
)

validation_results_df

In [ ]:
from sklearn.model_selection import train_test_split

#Using a smaller stratified sample from the training set for faster hyperparameter tuning
#The validation and final test sets are not used here

X_tune, _, y_tune, _ = train_test_split(
    X_train_final,
    y_train_final,
    train_size=250000,
    random_state=RANDOM_STATE,
    stratify=y_train_final
)

print("Tuning subset:", X_tune.shape)

print("\nTuning subset class counts:")
print(y_tune.value_counts())

print("\nTuning subset distribution:")
print(y_tune.value_counts(normalize=True).mul(100).round(2))

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

#Tuning HistGradientBoosting using cross-validation on the training subset

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

hgb_tuning_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", HistGradientBoostingClassifier(
        random_state=RANDOM_STATE
    ))
])

hgb_param_grid = {
    "classifier__learning_rate": [0.05, 0.1],
    "classifier__max_iter": [100, 200],
    "classifier__max_leaf_nodes": [31, 63],
    "classifier__l2_regularization": [0.0, 0.1]
}

hgb_grid = GridSearchCV(
    estimator=hgb_tuning_pipeline,
    param_grid=hgb_param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

hgb_grid.fit(X_tune, y_tune)

print("Best HistGradientBoosting parameters:")
print(hgb_grid.best_params_)

print("\nBest CV macro-F1:")
print(hgb_grid.best_score_)

In [ ]:
rf_tuning_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_param_grid = {
    "classifier__n_estimators": [100],
    "classifier__max_depth": [None, 40],
    "classifier__min_samples_split": [2, 5],
    "classifier__min_samples_leaf": [1]
}

rf_grid = GridSearchCV(
    estimator=rf_tuning_pipeline,
    param_grid=rf_param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_tune, y_tune)

print("Best Random Forest parameters:")
print(rf_grid.best_params_)

print("\nBest CV macro-F1:")
print(rf_grid.best_score_)

In [ ]:
#Training the tuned HGB and RF models on the full training set, then checking them on validation data

best_hgb_model = hgb_grid.best_estimator_
best_rf_model = rf_grid.best_estimator_

print("Training tuned HistGradientBoosting on full training set...")
best_hgb_model.fit(X_train_final, y_train_final)

hgb_val_pred = best_hgb_model.predict(X_val_final)
hgb_val_scores = get_attack_scores(best_hgb_model, X_val_final)

tuned_hgb_val_results = evaluate_predictions(
    y_val_final,
    hgb_val_pred,
    y_score=hgb_val_scores,
    dataset_name="Validation - Tuned HistGradientBoosting"
)

print("\n" + "=" * 80)
print("Training tuned Random Forest on full training set...")
best_rf_model.fit(X_train_final, y_train_final)

rf_val_pred = best_rf_model.predict(X_val_final)
rf_val_scores = get_attack_scores(best_rf_model, X_val_final)

tuned_rf_val_results = evaluate_predictions(
    y_val_final,
    rf_val_pred,
    y_score=rf_val_scores,
    dataset_name="Validation - Tuned Random Forest"
)

tuned_validation_results_df = pd.DataFrame([
    {**tuned_hgb_val_results, "model": "Tuned HistGradientBoosting"},
    {**tuned_rf_val_results, "model": "Tuned Random Forest"}
]).sort_values(by="attack_f1", ascending=False)

tuned_validation_results_df.to_csv(
    RESULTS_DIR / "tuned_validation_results_clean.csv",
    index=False
)

tuned_validation_results_df

In [ ]:
#Tuning the classification threshold on the validation set only
def threshold_tuning(y_true, attack_scores, model_name, thresholds=np.arange(0.01, 0.91, 0.02)):
    rows = []

    for threshold in thresholds:
        preds = np.where(attack_scores >= threshold, "ATTACK", "BENIGN")

        rows.append({
            "model": model_name,
            "threshold": threshold,
            "accuracy": accuracy_score(y_true, preds),
            "macro_f1": f1_score(y_true, preds, average="macro", zero_division=0),
            "attack_precision": precision_score(y_true, preds, pos_label="ATTACK", zero_division=0),
            "attack_recall": recall_score(y_true, preds, pos_label="ATTACK", zero_division=0),
            "attack_f1": f1_score(y_true, preds, pos_label="ATTACK", zero_division=0)
        })

    return pd.DataFrame(rows).sort_values(by="attack_f1", ascending=False)


hgb_threshold_results = threshold_tuning(
    y_val_final,
    hgb_val_scores,
    "Tuned HistGradientBoosting"
)

rf_threshold_results = threshold_tuning(
    y_val_final,
    rf_val_scores,
    "Tuned Random Forest"
)

threshold_results = pd.concat([
    hgb_threshold_results,
    rf_threshold_results
]).sort_values(by="attack_f1", ascending=False)

threshold_results.to_csv(
    RESULTS_DIR / "validation_threshold_tuning_clean.csv",
    index=False
)

threshold_results.head(20)

In [ ]:
#Saving the best validation thresholds for the final test evaluation
best_hgb_threshold = hgb_threshold_results.iloc[0]["threshold"]
best_rf_threshold = rf_threshold_results.iloc[0]["threshold"]

print("Best HGB threshold:", round(best_hgb_threshold, 2))
print("Best RF threshold:", round(best_rf_threshold, 2))

print("\nBest HGB validation threshold row:")
display(hgb_threshold_results.head(1))

print("\nBest RF validation threshold row:")
display(rf_threshold_results.head(1))

In [ ]:
#Evaluating the tuned models on the untouched final random test set
#The thresholds used here were selected from the validation set only

hgb_test_scores = get_attack_scores(best_hgb_model, X_test_final)
hgb_test_preds = np.where(
    hgb_test_scores >= best_hgb_threshold,
    "ATTACK",
    "BENIGN"
)

final_hgb_test_results = evaluate_predictions(
    y_test_final,
    hgb_test_preds,
    y_score=hgb_test_scores,
    dataset_name="Final Test - Tuned HistGradientBoosting"
)

final_hgb_test_results["model"] = "Tuned HistGradientBoosting"
final_hgb_test_results["threshold"] = round(best_hgb_threshold, 2)


rf_test_scores = get_attack_scores(best_rf_model, X_test_final)
rf_test_preds = np.where(
    rf_test_scores >= best_rf_threshold,
    "ATTACK",
    "BENIGN"
)

final_rf_test_results = evaluate_predictions(
    y_test_final,
    rf_test_preds,
    y_score=rf_test_scores,
    dataset_name="Final Test - Tuned Random Forest"
)

final_rf_test_results["model"] = "Tuned Random Forest"
final_rf_test_results["threshold"] = round(best_rf_threshold, 2)


final_test_results_clean = pd.DataFrame([
    final_hgb_test_results,
    final_rf_test_results
]).sort_values(by="attack_f1", ascending=False)

final_test_results_clean.to_csv(
    RESULTS_DIR / "final_test_results_clean.csv",
    index=False
)

final_test_results_clean

In [ ]:
#Checking the actual error counts for the best final test model

cm_hgb_final = confusion_matrix(
    y_test_final,
    hgb_test_preds,
    labels=["BENIGN", "ATTACK"]
)

print("Labels: BENIGN, ATTACK")
print(cm_hgb_final)

tn, fp, fn, tp = cm_hgb_final.ravel()

print("False positives:", fp)
print("False negatives:", fn)
print("Total wrong:", fp + fn)
print("Total test rows:", len(y_test_final))
print("Error rate:", (fp + fn) / len(y_test_final))

In [ ]:
#Checking that no obvious target-related columns are included as features
for col in feature_columns:
    if "label" in col.lower() or "attack" in col.lower() or "class" in col.lower():
        print(col)

In [ ]:
#Checking which source files are included in the random final test set
test_source_files = df_model.loc[X_test_final.index, "source_file"].value_counts()
print(test_source_files)

In [ ]:
#Creating a harder held-out Friday split to test day/scenario generalisation
test_files = [
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv"
]

train_day_df = df_model[~df_model["source_file"].isin(test_files)].copy()
test_day_df = df_model[df_model["source_file"].isin(test_files)].copy()

X_train_day = train_day_df[feature_columns]
y_train_day = train_day_df["binary_label"]

X_test_day = test_day_df[feature_columns]
y_test_day = test_day_df["binary_label"]

print("Held-out Friday train:", X_train_day.shape)
print("Held-out Friday test:", X_test_day.shape)

print("\nTrain distribution:")
print(y_train_day.value_counts())
print(y_train_day.value_counts(normalize=True).mul(100).round(2))

print("\nFriday test distribution:")
print(y_test_day.value_counts())
print(y_test_day.value_counts(normalize=True).mul(100).round(2))

In [ ]:
#Held-out Friday robustness evaluation
#Training on Monday-Thursday and testing on Friday only

heldout_models = {
    "Held-out Logistic Regression": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=500,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),

    "Held-out HistGradientBoosting": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", HistGradientBoostingClassifier(
            l2_regularization=0.1,
            learning_rate=0.1,
            max_iter=200,
            max_leaf_nodes=31,
            random_state=RANDOM_STATE
        ))
    ]),

    "Held-out Random Forest": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            max_depth=None,
            min_samples_split=5,
            min_samples_leaf=1,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])
}

heldout_results = []

for model_name, model in heldout_models.items():
    print("\n" + "=" * 80)
    print(f"Training {model_name}")
    print("=" * 80)

    model.fit(X_train_day, y_train_day)

    day_scores = get_attack_scores(model, X_test_day)

    #Using the default 0.50 threshold for a fair robustness comparison
    day_preds = np.where(day_scores >= 0.50, "ATTACK", "BENIGN")

    result = evaluate_predictions(
        y_test_day,
        day_preds,
        y_score=day_scores,
        dataset_name=model_name
    )

    result["model"] = model_name
    result["threshold"] = 0.50
    heldout_results.append(result)

heldout_robustness_results = pd.DataFrame(heldout_results).sort_values(
    by="attack_f1",
    ascending=False
)

heldout_robustness_results.to_csv(
    RESULTS_DIR / "heldout_friday_robustness_results_clean.csv",
    index=False
)

heldout_robustness_results

In [ ]:
#Checking threshold sensitivity on the held-out Friday test set
#This is for analysis only, not for selecting the main held-out result

def threshold_tuning_fast(y_true, attack_scores, model_name, thresholds=np.arange(0.01, 0.91, 0.02)):
    rows = []
    y_attack = (np.asarray(y_true) == "ATTACK")

    for threshold in thresholds:
        pred_attack = attack_scores >= threshold

        tp = np.sum(pred_attack & y_attack)
        fp = np.sum(pred_attack & ~y_attack)
        fn = np.sum(~pred_attack & y_attack)
        tn = np.sum(~pred_attack & ~y_attack)

        accuracy = (tp + tn) / len(y_true)

        attack_precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        attack_recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        attack_f1 = (
            2 * attack_precision * attack_recall / (attack_precision + attack_recall)
            if (attack_precision + attack_recall) > 0 else 0
        )

        benign_precision = tn / (tn + fn) if (tn + fn) > 0 else 0
        benign_recall = tn / (tn + fp) if (tn + fp) > 0 else 0
        benign_f1 = (
            2 * benign_precision * benign_recall / (benign_precision + benign_recall)
            if (benign_precision + benign_recall) > 0 else 0
        )

        macro_f1 = (attack_f1 + benign_f1) / 2

        rows.append({
            "model": model_name,
            "threshold": round(threshold, 2),
            "accuracy": accuracy,
            "macro_f1": macro_f1,
            "attack_precision": attack_precision,
            "attack_recall": attack_recall,
            "attack_f1": attack_f1
        })

    return pd.DataFrame(rows).sort_values(by="attack_f1", ascending=False)


heldout_logreg_model = heldout_models["Held-out Logistic Regression"]
heldout_hgb_model = heldout_models["Held-out HistGradientBoosting"]
heldout_rf_model = heldout_models["Held-out Random Forest"]

heldout_logreg_scores = get_attack_scores(heldout_logreg_model, X_test_day)
heldout_hgb_scores = get_attack_scores(heldout_hgb_model, X_test_day)
heldout_rf_scores = get_attack_scores(heldout_rf_model, X_test_day)

heldout_logreg_threshold_results = threshold_tuning_fast(
    y_test_day,
    heldout_logreg_scores,
    "Held-out Logistic Regression"
)

heldout_hgb_threshold_results = threshold_tuning_fast(
    y_test_day,
    heldout_hgb_scores,
    "Held-out HistGradientBoosting"
)

heldout_rf_threshold_results = threshold_tuning_fast(
    y_test_day,
    heldout_rf_scores,
    "Held-out Random Forest"
)

heldout_threshold_results = pd.concat([
    heldout_logreg_threshold_results,
    heldout_hgb_threshold_results,
    heldout_rf_threshold_results
]).sort_values(by="attack_f1", ascending=False)

heldout_threshold_results.to_csv(
    RESULTS_DIR / "heldout_friday_threshold_sensitivity_results.csv",
    index=False
)

heldout_threshold_results.head(20)

In [ ]:
#Saving the best held-out threshold sensitivity result
#This is used only to analyse whether threshold choice explains the held-out drop

best_heldout_threshold_row = heldout_threshold_results.iloc[0]

print("Best held-out Friday threshold sensitivity result:")
display(best_heldout_threshold_row)

best_heldout_threshold_row.to_frame().T.to_csv(
    RESULTS_DIR / "best_heldout_friday_threshold_result.csv",
    index=False
)

In [ ]:
#Creating a summary table comparing random final testing and held-out Friday testing
summary_comparison = pd.DataFrame([
    {
        "evaluation": "Random final test",
        "model": "Tuned HistGradientBoosting",
        "threshold": round(best_hgb_threshold, 2),
        "accuracy": final_hgb_test_results["accuracy"],
        "attack_precision": final_hgb_test_results["attack_precision"],
        "attack_recall": final_hgb_test_results["attack_recall"],
        "attack_f1": final_hgb_test_results["attack_f1"],
        "interpretation": "Best in-distribution random-split model"
    },
    {
        "evaluation": "Held-out Friday default threshold",
        "model": "Logistic Regression",
        "threshold": 0.50,
        "accuracy": heldout_robustness_results.loc[
            heldout_robustness_results["model"] == "Held-out Logistic Regression",
            "accuracy"
        ].iloc[0],
        "attack_precision": heldout_robustness_results.loc[
            heldout_robustness_results["model"] == "Held-out Logistic Regression",
            "attack_precision"
        ].iloc[0],
        "attack_recall": heldout_robustness_results.loc[
            heldout_robustness_results["model"] == "Held-out Logistic Regression",
            "attack_recall"
        ].iloc[0],
        "attack_f1": heldout_robustness_results.loc[
            heldout_robustness_results["model"] == "Held-out Logistic Regression",
            "attack_f1"
        ].iloc[0],
        "interpretation": "Best robustness/generalisation model"
    },
    {
        "evaluation": "Held-out Friday threshold sensitivity",
        "model": best_heldout_threshold_row["model"],
        "threshold": best_heldout_threshold_row["threshold"],
        "accuracy": best_heldout_threshold_row["accuracy"],
        "attack_precision": best_heldout_threshold_row["attack_precision"],
        "attack_recall": best_heldout_threshold_row["attack_recall"],
        "attack_f1": best_heldout_threshold_row["attack_f1"],
        "interpretation": "Best attack-F1 after threshold sensitivity analysis"
    }
])

summary_comparison.to_csv(
    RESULTS_DIR / "random_vs_heldout_summary_comparison.csv",
    index=False
)

summary_comparison

In [ ]:
#Saving the confusion matrix for the random final test HGB model

cm_hgb_final = confusion_matrix(
    y_test_final,
    hgb_test_preds,
    labels=["BENIGN", "ATTACK"]
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_hgb_final,
    display_labels=["BENIGN", "ATTACK"]
)

disp.plot(values_format="d")
plt.title("Tuned HGB Confusion Matrix - Random Final Test")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "hgb_random_final_test_confusion_matrix.png", dpi=300)
plt.show()

cm_hgb_final

In [ ]:
#Saving the confusion matrix for held-out Friday Logistic Regression

heldout_logreg_best_preds = np.where(
    heldout_logreg_scores >= 0.51,
    "ATTACK",
    "BENIGN"
)

cm_logreg_heldout = confusion_matrix(
    y_test_day,
    heldout_logreg_best_preds,
    labels=["BENIGN", "ATTACK"]
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_logreg_heldout,
    display_labels=["BENIGN", "ATTACK"]
)

disp.plot(values_format="d")
plt.title("Logistic Regression Confusion Matrix - Held-out Friday")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "logreg_heldout_friday_confusion_matrix.png", dpi=300)
plt.show()
plt.close()

cm_logreg_heldout

In [ ]:
#Saving the precision-recall curve for the tuned HGB random final test model
from sklearn.metrics import PrecisionRecallDisplay

PrecisionRecallDisplay.from_predictions(
    (y_test_final == "ATTACK").astype(int),
    hgb_test_scores,
    name="Tuned HGB - Random Final Test"
)

plt.title("Precision-Recall Curve - Tuned HGB Random Final Test")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "hgb_random_final_test_pr_curve.png", dpi=300)
plt.show()

In [ ]:
#Saving the precision-recall curve for held-out Friday Logistic Regression
PrecisionRecallDisplay.from_predictions(
    (y_test_day == "ATTACK").astype(int),
    heldout_logreg_scores,
    name="Logistic Regression - Held-out Friday"
)

plt.title("Precision-Recall Curve - Logistic Regression Held-out Friday")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "logreg_heldout_friday_pr_curve.png", dpi=300)
plt.show()
plt.close()

In [ ]:
#Saving the final trained models

joblib.dump(
    best_hgb_model,
    MODEL_DIR / "tuned_histgradientboosting_final_model.joblib"
)

joblib.dump(
    best_rf_model,
    MODEL_DIR / "tuned_random_forest_final_model.joblib"
)

joblib.dump(
    heldout_logreg_model,
    MODEL_DIR / "logistic_regression_heldout_friday_model.joblib"
)

print("Models saved:")
print(MODEL_DIR / "tuned_histgradientboosting_final_model.joblib")
print(MODEL_DIR / "tuned_random_forest_final_model.joblib")
print(MODEL_DIR / "logistic_regression_heldout_friday_model.joblib")

In [ ]:
#Reloading the saved HGB model and checking it reproduces the final random test result

loaded_hgb_model = joblib.load(
    MODEL_DIR / "tuned_histgradientboosting_final_model.joblib"
)

loaded_hgb_scores = get_attack_scores(loaded_hgb_model, X_test_final)

loaded_hgb_preds = np.where(
    loaded_hgb_scores >= best_hgb_threshold,
    "ATTACK",
    "BENIGN"
)

loaded_hgb_results = evaluate_predictions(
    y_test_final,
    loaded_hgb_preds,
    y_score=loaded_hgb_scores,
    dataset_name="Loaded HGB - Random Final Test"
)

assert abs(loaded_hgb_results["attack_f1"] - final_hgb_test_results["attack_f1"]) < 1e-6

print("Loaded HGB reproduces the saved final test result.")
plt.close()

In [ ]:
#Reloading the saved Logistic Regression model and checking the held-out Friday result

loaded_logreg_model = joblib.load(
    MODEL_DIR / "logistic_regression_heldout_friday_model.joblib"
)

loaded_logreg_scores = get_attack_scores(loaded_logreg_model, X_test_day)

loaded_logreg_preds = np.where(
    loaded_logreg_scores >= 0.51,
    "ATTACK",
    "BENIGN"
)

loaded_logreg_results = evaluate_predictions(
    y_test_day,
    loaded_logreg_preds,
    y_score=loaded_logreg_scores,
    dataset_name="Loaded Logistic Regression - Held-out Friday"
)

assert abs(loaded_logreg_results["attack_f1"] - best_heldout_threshold_row["attack_f1"]) < 1e-6

print("Loaded Logistic Regression reproduces the held-out Friday result.")

In [ ]:
#Saving Random Forest feature importance for model interpretation

rf_classifier = best_rf_model.named_steps["classifier"]

feature_importance_df = pd.DataFrame({
    "feature": feature_columns,
    "importance": rf_classifier.feature_importances_
}).sort_values(by="importance", ascending=False)

feature_importance_df.to_csv(
    RESULTS_DIR / "random_forest_feature_importance_clean.csv",
    index=False
)

feature_importance_df.head(25)

In [ ]:
#Plotting the top 15 Random Forest feature importances

top_features = feature_importance_df.head(15).sort_values(by="importance")

plt.figure(figsize=(8, 6))
plt.barh(top_features["feature"], top_features["importance"])
plt.xlabel("Feature importance")
plt.title("Top 15 Random Forest Feature Importances")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "random_forest_top15_feature_importance.png", dpi=300)
plt.show()
plt.close()

In [ ]:
#Checking the saved result, figure and model files
print("Result files:")
for file in sorted(RESULTS_DIR.iterdir()):
    print(file.name)

print("\nFigure files:")
for file in sorted(FIGURE_DIR.iterdir()):
    print(file.name)

print("\nModel files:")
for file in sorted(MODEL_DIR.iterdir()):
    print(file.name)

In [ ]:
#Saving attack-type distribution plot for the report

import matplotlib.pyplot as plt

attack_type_distribution = df_model["Label"].value_counts().reset_index()
attack_type_distribution.columns = ["label", "count"]

plt.figure(figsize=(10, 6))
plt.barh(
    attack_type_distribution["label"][::-1],
    attack_type_distribution["count"][::-1]
)

plt.xlabel("Number of records")
plt.ylabel("Original label")
plt.title("Attack-type distribution after deduplication")
plt.tight_layout()

plt.savefig(FIGURE_DIR / "attack_type_distribution.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()